# 02b - Validation independante des resultats (Partie 2/3)
Calcul du chiffre d'affaires par ville, de la duree moyenne par ville/type et du top 5 de **trois facons** : Python pur (`data/validate_data.py`), RDD et DataFrame. Les trois doivent concorder.

In [1]:
import sys, time
sys.path.append('/home/jovyan/work/pipeline'); sys.path.append('/home/jovyan/work/data')
from pedali_spark import get_spark
spark = get_spark('pedali-02b-validation')
sc = spark.sparkContext
print('Spark', spark.version, '| master', sc.master, '| defaultParallelism', sc.defaultParallelism)

import json
from pedali_spark import get_spark
import rdd_analysis, df_analysis
from validate_data import reference
from pyspark.sql import functions as F

Spark 3.5.3 | master spark://spark-master:7077 | defaultParallelism 4


## 1. Reference Python pur (sans Spark)

In [2]:
ref = reference('/home/jovyan/work/data/trips_history.jsonl', '/home/jovyan/work/data/stations.json')
for k, v in ref.items(): print(k, ':', v)

evenements : 50000
doublons : 1003
stations : 400
statuts : {'RESERVED': 16729, 'UNLOCKED': 15542, 'ENDED': 14880, 'CANCELLED': 1188, 'FAULT_REPORTED': 658}
trajets_termines : 14880
chiffre_affaires_par_ville : {'Casablanca': 75430.27, 'Marrakech': 51959.36, 'Rabat': 43473.12, 'Tanger': 42600.26}
duree_moyenne_s_par_ville_type : {'Casablanca/BIKE': 1089.1, 'Casablanca/EBIKE': 1096.3, 'Casablanca/SCOOTER': 1089.7, 'Marrakech/BIKE': 1093.5, 'Marrakech/EBIKE': 1114.3, 'Marrakech/SCOOTER': 1099.1, 'Rabat/BIKE': 1089.2, 'Rabat/EBIKE': 1098.0, 'Rabat/SCOOTER': 1088.8, 'Tanger/BIKE': 1088.3, 'Tanger/EBIKE': 1108.1, 'Tanger/SCOOTER': 1100.3}
top5_stations_depart : [('S-CAS-005', 343), ('S-CAS-097', 93), ('S-CAS-096', 87), ('S-MAR-007', 87), ('S-CAS-084', 85)]
inconnues : 0


## 2. Controle qualite (schema explicite)

In [3]:
from pedali_schema import read_trips, read_stations, quality_report
trips_raw = read_trips(spark, 's3a://lake/bronze/history/trips_history.jsonl')
stations = read_stations(spark, 's3a://lake/bronze/history/stations.json')
trips_raw.printSchema()
for k, v in quality_report(trips_raw, stations).items(): print(f'{k:42s} {v}')

root
 |-- trip_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- start_station: string (nullable = true)
 |-- end_station: string (nullable = true)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- duration_s: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- battery_pct: integer (nullable = true)
 |-- price_mad: double (nullable = true)
 |-- event_time: timestamp (nullable = true)



evenements                                 50000
doublons (meme trip_id + meme statut)      1003
station de depart inconnue                 0
ENDED sans duree/prix                      0
champs ENDED renseignes hors ENDED         0
duree negative                             0


## 3. RDD et DataFrame

In [4]:
res = rdd_analysis.run(spark)
stations_df, trips = df_analysis.load(spark)
ended = trips.filter("status = 'ENDED'")
df_ca = {r.city: round(r.ca, 2) for r in ended.groupBy('city').agg(F.sum('price_mad').alias('ca')).collect()}
df_dur = {f'{r.city}/{r.vehicle_type}': round(r.d, 1) for r in ended.groupBy('city', 'vehicle_type').agg(F.avg('duration_s').alias('d')).collect()}
df_top = [(r.start_station, r['count']) for r in ended.groupBy('start_station').count().orderBy(F.desc('count'), 'start_station').limit(5).collect()]

## 4. Les trois resultats concordent

In [5]:
assert ref['chiffre_affaires_par_ville'] == res['chiffre_affaires_par_ville'] == dict(sorted(df_ca.items())), 'CA different'
assert ref['duree_moyenne_s_par_ville_type'] == res['duree_moyenne_s_par_ville_type'] == dict(sorted(df_dur.items())), 'duree differente'
assert ref['top5_stations_depart'] == res['top5_stations_depart'] == df_top, 'top 5 different'
assert ref['trajets_termines'] == res['trajets_termines'] == ended.count()
print('OK : Python pur == RDD == DataFrame (CA par ville, duree moyenne par ville/type, top 5, nombre de trajets)')
print('CA par ville :', ref['chiffre_affaires_par_ville'])
print('Top 5       :', ref['top5_stations_depart'])

OK : Python pur == RDD == DataFrame (CA par ville, duree moyenne par ville/type, top 5, nombre de trajets)
CA par ville : {'Casablanca': 75430.27, 'Marrakech': 51959.36, 'Rabat': 43473.12, 'Tanger': 42600.26}
Top 5       : [('S-CAS-005', 343), ('S-CAS-097', 93), ('S-CAS-096', 87), ('S-MAR-007', 87), ('S-CAS-084', 85)]


In [6]:
spark.stop()